# LLM Safety Eval Pipeline — Colab runner

Runs a HuggingFace model against jailbreak safety benchmarks (JailbreakBench, HarmBench, StrongREJECT) and pushes transparent results (every prompt, response, and score) to your GitHub repo.

**Before running:**
1. Runtime -> Change runtime type -> **T4 GPU** (free tier is fine for small models).
2. In the left sidebar, click the key icon (Secrets) and add:
   - `GITHUB_TOKEN` — a GitHub personal access token so this notebook can push results. Use a **fine-grained token scoped to just this one repo** (Contents: read/write is enough), not a classic token with blanket `repo` scope -- if it ever leaks, this limits the damage to this repo alone.
   - `HF_TOKEN` — optional, only needed if you use HarmBench (it's a gated dataset).
   - `ANTHROPIC_API_KEY` or `OPENAI_API_KEY` — optional, only needed for `llm_judge`; add the key for the provider selected in cell 1.
3. **Before you ever save/share/commit this notebook file itself**, clear its outputs first (Edit -> Clear all outputs). None of the cells below print secrets deliberately, but if a git command ever fails, error output could otherwise end up saved into a public place. Clearing outputs is a cheap habit that closes that door entirely.

In [ ]:
# 1. Configure your run here
GITHUB_REPO_URL = "https://github.com/<your-username>/<your-repo>.git"  # <-- edit me
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # small + popular + Colab-T4-friendly. Swap for any HF model id.
EVALS = "jailbreakbench,strongreject"      # add harmbench once you've accepted its HF terms
N_PROMPTS = 15                              # 0 = full benchmark. Start small to work out kinks.
SCORER = "rule_based"                       # free. Use "llm_judge" for higher fidelity (costs a little API credit).
JUDGE_PROVIDER = "anthropic"                 # used only with llm_judge: "anthropic" or "openai".
JUDGE_MODEL = "claude-sonnet-4-6"             # used only with llm_judge; e.g. "gpt-4.1-mini" for OpenAI.

# Commit identity -- MUST match a verified email on your GitHub account, or
# Vercel/GitHub will silently block the deploy ("could not associate the
# committer with a GitHub user"). Easiest: use your GitHub noreply email,
# found at https://github.com/settings/emails -- looks like
# "12345678+yourusername@users.noreply.github.com".
GIT_AUTHOR_NAME = "<your-github-username>"    # <-- edit me
GIT_AUTHOR_EMAIL = "<id>+<your-github-username>@users.noreply.github.com"  # <-- edit me

In [ ]:
# 2. Secrets (set these in Colab's Secrets panel, not typed here)
from google.colab import userdata
import os

os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass
try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass

In [ ]:
# 3. Clone your repo (must already exist on GitHub -- see README for one-time setup) and install deps
#
# NOTE: the token is passed as a one-off HTTP header (-c http.extraheader), not
# embedded in the URL. This matters: if a clone/push ever fails, git prints the
# URL it tried in its error message -- if the token were embedded in that URL,
# it would leak in plaintext into this cell's saved output. The header approach
# keeps the token out of any URL git might echo back, and it's never written to
# .git/config either.
repo_name = GITHUB_REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
AUTH_HEADER = f"AUTHORIZATION: bearer {os.environ['GITHUB_TOKEN']}"

!git -c http.extraheader="{AUTH_HEADER}" clone {GITHUB_REPO_URL}
%cd {repo_name}
!pip install -q -r requirements.txt

In [ ]:
# 4. Run the eval(s)
!python src/run_eval.py --model "{MODEL_ID}" --eval "{EVALS}" --n {N_PROMPTS} --scorer "{SCORER}" --judge-provider "{JUDGE_PROVIDER}" --judge-model "{JUDGE_MODEL}"

In [ ]:
# 5. Regenerate the report, then commit + push everything back to GitHub.
#
# report.py runs first, every time -- it rescans all of results/*.json (including
# results from earlier sessions, since cell 3's clone pulls whatever's already been
# pushed) and rewrites README.md + leaderboard.json + per-run transcripts to match.
# This step used to be a separate cell; folding it in here means it's impossible to
# push new eval results without also refreshing the aggregated report to match.
!python src/report.py
!git config user.email "{GIT_AUTHOR_EMAIL}"
!git config user.name "{GIT_AUTHOR_NAME}"
!git add results/
!git commit -m "eval run: {MODEL_ID} on {EVALS} ({SCORER})"
!git -c http.extraheader="{AUTH_HEADER}" push

---
## Batch mode: iterate many models

Cells 1-5 above are for a single quick run. To evaluate a whole list of models in one go, edit `models.txt` in the repo (one HF model id per line -- a starter list of small, popular, Colab-friendly models is already there) and run the cell below instead. **This is the recommended path for a long `models.txt`.**

**Commit + push happens after *each model*, not at the end.** As soon as a model finishes its evals, `run_batch.py --push` regenerates the report and pushes the new results straight to GitHub before moving to the next model. So if the runtime disconnects, you crash, or you hit a Colab timeout, every model completed *so far* is already safely pushed -- you never lose more than the one model that was mid-run.

**It's fully resumable.** Just re-run this cell after any interruption:
- Models with complete, valid results for every requested eval are **skipped**.
- A model whose result file is **missing, corrupt, or incomplete** (e.g. a half-written file from a killed process or a broken upload) is treated as *not done* and re-run. Within that model, only the specific eval(s) that are missing/broken are redone -- finished evals aren't repeated.
- Models that fail to load (gated without a token, too large for the GPU, etc.) are logged to `results/_failures.log` and skipped rather than stopping the batch.

Because of this, one bad or partial result can never wedge the pipeline: `report.py` skips any unreadable file (logging it) instead of crashing, and the next run simply regenerates it.

In [ ]:
# Batch run across every model in models.txt, pushing after each one.
os.environ["GIT_AUTHOR_NAME"] = GIT_AUTHOR_NAME
os.environ["GIT_AUTHOR_EMAIL"] = GIT_AUTHOR_EMAIL
os.environ["AUTH_HEADER"] = AUTH_HEADER

!python src/run_batch.py --models-file models.txt --eval "{EVALS}" --n {N_PROMPTS} --scorer "{SCORER}" --judge-provider "{JUDGE_PROVIDER}" --judge-model "{JUDGE_MODEL}" --push

In [ ]:
# Optional: preview what a batch run would do without actually running anything
# (useful after editing models.txt, to sanity-check the list and see what's
# already done vs. still queued).
!python src/run_batch.py --models-file models.txt --eval "{EVALS}" --n {N_PROMPTS} --scorer "{SCORER}" --judge-provider "{JUDGE_PROVIDER}" --judge-model "{JUDGE_MODEL}" --dry-run